**Weighted linear regression (WLS)** is used when ordinary least squares (OLS) regression assumptions are violated, specifically when data exhibits heteroscedasticity (non-constant variance of residuals). It assigns weights to observations to give more importance to reliable, low-variance data points and less weight to noisy ones, ensuring efficient estimates.

A mean based on a million records (in 2022) is far more reliable than one based on 29,000 (2017). If you treated them equally, the noisy early years would distort your trend line just as much as the stable later years. We are putting more weights on years with a higher number of observations. 


So you use weights = n — the sample size for each year — meaning years with more data pull the regression line harder toward them.

Weighted OLS minimises: Σ wᵢ × (residual²)

Where weighted linear regression assumes your data follows a straight line with normally distributed errors, **Mann-Kendall is non-parametric** — it makes no assumptions about the shape of the trend or the distribution of your data. Instead of asking "what is the slope?", it asks a simpler question:

Does the series tend to go up over time, or down — more than you'd expect by chance?

**How MK works** :

It looks at every possible pair of years (i, j) where j comes after i, and just asks: did the value go up or down?

Each pair votes:

* +1 if later year is higher (upward)
* −1 if later year is lower (downward)
* 0 if equal (tie)

S is the total score. With 6 years you have 6×5/2 = 15 pairs, so S ranges from −15 to +15. (we're working with permutations)

In [ ]:
import numpy as np
from scipy import stats

# Data
years = np.array([2017, 2018, 2019, 2020, 2021, 2022])

native_mean   = np.array([41.865980, 45.332769, 44.491083, 41.682730, 40.224726, 40.813832])
native_n      = np.array([29046, 340439, 494781, 713935, 970514, 1087847])

migrating_mean = np.array([90.610106, 89.488594, 81.937664, 77.234132, 79.641804, 70.922299])
migrating_n    = np.array([59795, 70218, 91941, 104121, 146936, 195581])

# ─────────────────────────────────────────────
# 1. WEIGHTED LINEAR REGRESSION
# ─────────────────────────────────────────────



def weighted_regression(years, means, weights):
    """
    Weighted OLS: minimises sum of w_i * (y_i - (a + b*x_i))^2
    Returns slope, intercept, R², and p-value for the slope.
    """
    w = weights / weights.sum()           # normalise weights
    x = years - years.mean()              # centre year to reduce collinearity

    x_bar = np.average(x, weights=w)
    y_bar = np.average(means, weights=w)

    Sxx = np.sum(w * (x - x_bar)**2)
    Sxy = np.sum(w * (x - x_bar) * (means - y_bar))
    Syy = np.sum(w * (means - y_bar)**2)

    slope     = Sxy / Sxx
    intercept = y_bar - slope * x_bar

    fitted    = intercept + slope * x
    residuals = means - fitted
    SS_res    = np.sum(w * residuals**2)
    SS_tot    = Syy
    r2        = 1 - SS_res / SS_tot

    # Standard error of slope; using n-2 degrees of freedom
    n      = len(years)
    s2     = SS_res / (n - 2)
    se_b   = np.sqrt(s2 / Sxx)
    t_stat = slope / se_b
    p_val  = 2 * stats.t.sf(abs(t_stat), df=n - 2)

    return slope, intercept, r2, t_stat, p_val, se_b


native_slope, native_intercept, native_r2, native_t, native_p, native_se = \
    weighted_regression(years, native_mean, native_n)

migr_slope, migr_intercept, migr_r2, migr_t, migr_p, migr_se = \
    weighted_regression(years, migrating_mean, migrating_n)

print("=" * 55)
print("WEIGHTED LINEAR REGRESSION")
print("=" * 55)
for label, slope, intercept, r2, t, p, se in [
    ("Native",    native_slope,  native_intercept,  native_r2,  native_t,  native_p,  native_se),
    ("Migrating", migr_slope,    migr_intercept,    migr_r2,    migr_t,    migr_p,    migr_se),
]:
    print(f"\n{label}")
    print(f"  Slope       : {slope:+.4f} km/year  (SE = {se:.4f})")
    print(f"  Intercept   : {intercept:.4f}")
    print(f"  R²          : {r2:.4f}")
    print(f"  t-statistic : {t:.4f}")
    print(f"  p-value     : {p:.4f}  {'*** significant' if p < 0.05 else 'not significant'}")

# Interaction test: are the two slopes significantly different?
diff        = native_slope - migr_slope
se_diff     = np.sqrt(native_se**2 + migr_se**2) #  standard error propagation
t_diff      = diff / se_diff # t test cause the regression slopes each have only 6 data points (2017–2022) (low sample size)
p_diff      = 2 * stats.t.sf(abs(t_diff), df=(len(years) - 2) * 2) # getting the p value

print(f"\nSlope difference (native − migrating): {diff:+.4f} km/year")
print(f"  SE of difference : {se_diff:.4f}")
print(f"  t-statistic      : {t_diff:.4f}")
print(f"  p-value          : {p_diff:.4f}  {'*** significant' if p_diff < 0.05 else 'not significant'}")

# ─────────────────────────────────────────────
# 2. MANN-KENDALL TREND TEST
# ─────────────────────────────────────────────

def mann_kendall(x):
    """
    Manual Mann-Kendall test.
    S  = sum of sign(x_j - x_i) for all i < j
    Var(S) uses the standard formula (no tied correction needed for continuous data).
    Returns S, tau, z-statistic, and p-value.
    """
    n = len(x)
    s = 0
    for i in range(n - 1):
        for j in range(i + 1, n):
            s += np.sign(x[j] - x[i])

    # Variance of S under H0
    var_s = n * (n - 1) * (2 * n + 5) / 18 # the expected spread of S under the null hypothesis (no trend)

    # Continuity-corrected z
    if s > 0:
        z = (s - 1) / np.sqrt(var_s)
    elif s < 0:
        z = (s + 1) / np.sqrt(var_s)
    else:
        z = 0.0

    p = 2 * stats.norm.sf(abs(z))

    # Kendall's tau
    tau = s / (n * (n - 1) / 2)

    return s, tau, z, p


native_s, native_tau, native_z, native_mk_p = mann_kendall(native_mean)
migr_s,   migr_tau,   migr_z,   migr_mk_p  = mann_kendall(migrating_mean)

print("\n" + "=" * 55)
print("MANN-KENDALL TREND TEST")
print("=" * 55)
for label, s, tau, z, p in [
    ("Native",    native_s, native_tau, native_z, native_mk_p),
    ("Migrating", migr_s,   migr_tau,   migr_z,   migr_mk_p),
]:
    direction = "downward" if tau < 0 else "upward"
    print(f"\n{label}")
    print(f"  S statistic : {s}")
    print(f"  Kendall tau : {tau:+.4f}  ({direction} trend)")
    print(f"  z-statistic : {z:.4f}")
    print(f"  p-value     : {p:.4f}  {'*** significant' if p < 0.05 else 'not significant'}")

WEIGHTED LINEAR REGRESSION

Native
  Slope       : -1.1354 km/year  (SE = 0.3388)
  Intercept   : 42.9147
  R²          : 0.7374
  t-statistic : -3.3511
  p-value     : 0.0285  *** significant

Migrating
  Slope       : -3.7972 km/year  (SE = 0.6974)
  Intercept   : 81.6631
  R²          : 0.8811
  t-statistic : -5.4446
  p-value     : 0.0055  *** significant

Slope difference (native − migrating): +2.6618 km/year
  SE of difference : 0.7754
  t-statistic      : 3.4329
  p-value          : 0.0089  *** significant

MANN-KENDALL TREND TEST

Native
  S statistic : -9.0
  Kendall tau : -0.6000  (downward trend)
  z-statistic : -1.5029
  p-value     : 0.1329  not significant

Migrating
  S statistic : -13.0
  Kendall tau : -0.8667  (downward trend)
  z-statistic : -2.2544
  p-value     : 0.0242  *** significant


#### better MK

The standard Mann-Kendall used a normal approximation to get the p-value — it assumed that S, under the null hypothesis, is approximately normally distributed. With only 6 data points that approximation is rough. There are only 15 pairs, so S can only take integer values in a narrow range, and the true distribution is lumpy and discrete, not smooth.


**The core idea here**
Instead of approximating, this version asks: if there were truly no trend, what's the exact probability of seeing an S this extreme?
It answers that by brute force — enumerate every possible ordering of your 6 values and compute S for each one. With 6 data points there are 6! = 720 permutations. Each permutation represents a possible world where the same values occurred in a random order (the null hypothesis) 

this would be to see a random change in slope from i to j. we are constructing our sample space. then we have our actual values (slope change from year 2017-2018) and we are seeing if that change is significant or just by chance.

In [ ]:
from itertools import combinations

def mann_kendall_exact(x):
    """
    Exact Mann-Kendall p-value via full permutation of all n! orderings.
    Only feasible for small n (<=10 or so).
    """
    from itertools import permutations

    n = len(x)

    def compute_s(seq):
        s = 0
        for i in range(len(seq) - 1):
            for j in range(i + 1, len(seq)):
                s += np.sign(seq[j] - seq[i])
        return s

    observed_s = compute_s(x)

    # Generate all permutations and compute S for each
    all_s = [compute_s(perm) for perm in permutations(x)] # this is the brute-force way to get the exact distribution of S under the null hypothesis
    total = len(all_s)

    # Two-tailed: count how often |S_perm| >= |observed_s|
    p_exact = sum(1 for s in all_s if abs(s) >= abs(observed_s)) / total # "In what fraction of random orderings is S at least as extreme as what we actually observed?"



    tau = observed_s / (n * (n - 1) / 2)

    return observed_s, tau, p_exact


print("\n" + "=" * 55)
print("MANN-KENDALL — EXACT PERMUTATION P-VALUES")
print("=" * 55)
for label, data in [("Native", native_mean), ("Migrating", migrating_mean)]:
    s, tau, p = mann_kendall_exact(data)
    direction = "downward" if tau < 0 else "upward"
    print(f"\n{label}")
    print(f"  S statistic : {s}")
    print(f"  Kendall tau : {tau:+.4f}  ({direction} trend)")
    print(f"  p-value     : {p:.4f}  {'*** significant' if p < 0.05 else 'not significant'}")


MANN-KENDALL — EXACT PERMUTATION P-VALUES

Native
  S statistic : -9.0
  Kendall tau : -0.6000  (downward trend)
  p-value     : 0.1361  not significant

Migrating
  S statistic : -13.0
  Kendall tau : -0.8667  (downward trend)
  p-value     : 0.0167  *** significant


interpreting the results from the first cell: Weighted linear regression revealed a statistically significant negative trend in mean distance to the nearest city for both native birds (slope = −1.14 km/year, p = 0.029) and migrating birds (slope = −3.80 km/year, p = 0.006) over the 2017–2022 period, indicating that both groups are being observed increasingly closer to urban areas over time. A Mann-Kendall trend test corroborated the migrating birds result (τ = −0.87, p = 0.017), but did not reach significance for native birds (τ = −0.60, p = 0.136), suggesting that while the overall linear direction is downward, the native bird trend is not strictly monotonic across years. Critically, the difference in slopes between the two groups was itself statistically significant (p = 0.009), indicating that migrating birds are approaching urban areas at a rate approximately three times faster than native birds.

## robustness checks

In [9]:
import numpy as np
from scipy import stats
from itertools import permutations

years = np.array([2017, 2018, 2019, 2020, 2021, 2022])

native_median   = np.array([32.825, 28.570, 28.860, 27.000, 27.090, 28.290])
native_n        = np.array([29046, 340439, 494781, 713935, 970514, 1087847])

migrating_median = np.array([35.56, 35.54, 35.73, 35.62, 36.12, 35.70])
migrating_n      = np.array([59795, 70218, 91941, 104121, 146936, 195581])

# ─────────────────────────────────────────────
# 1. WEIGHTED LINEAR REGRESSION ON MEDIAN
# ─────────────────────────────────────────────

def weighted_regression(years, values, weights):
    w = weights / weights.sum()
    x = years - years.mean()

    x_bar = np.average(x, weights=w)
    y_bar = np.average(values, weights=w)

    Sxx = np.sum(w * (x - x_bar)**2)
    Sxy = np.sum(w * (x - x_bar) * (values - y_bar))
    Syy = np.sum(w * (values - y_bar)**2)

    slope     = Sxy / Sxx
    intercept = y_bar - slope * x_bar

    fitted    = intercept + slope * x
    residuals = values - fitted
    SS_res    = np.sum(w * residuals**2)
    r2        = 1 - SS_res / Syy

    n      = len(years)
    s2     = SS_res / (n - 2)
    se_b   = np.sqrt(s2 / Sxx)
    t_stat = slope / se_b
    p_val  = 2 * stats.t.sf(abs(t_stat), df=n - 2)

    return slope, intercept, r2, t_stat, p_val, se_b


native_slope, native_intercept, native_r2, native_t, native_p, native_se = \
    weighted_regression(years, native_median, native_n)

migr_slope, migr_intercept, migr_r2, migr_t, migr_p, migr_se = \
    weighted_regression(years, migrating_median, migrating_n)

print("=" * 55)
print("WEIGHTED LINEAR REGRESSION ON MEDIAN")
print("=" * 55)
for label, slope, intercept, r2, t, p, se in [
    ("Native",    native_slope, native_intercept, native_r2, native_t, native_p, native_se),
    ("Migrating", migr_slope,   migr_intercept,   migr_r2,   migr_t,   migr_p,   migr_se),
]:
    print(f"\n{label}")
    print(f"  Slope       : {slope:+.4f} km/year  (SE = {se:.4f})")
    print(f"  Intercept   : {intercept:.4f}")
    print(f"  R²          : {r2:.4f}")
    print(f"  t-statistic : {t:.4f}")
    print(f"  p-value     : {p:.4f}  {'*** significant' if p < 0.05 else 'not significant'}")

diff    = native_slope - migr_slope
se_diff = np.sqrt(native_se**2 + migr_se**2)
t_diff  = diff / se_diff
p_diff  = 2 * stats.t.sf(abs(t_diff), df=(len(years) - 2) * 2)

print(f"\nSlope difference (native − migrating): {diff:+.4f} km/year")
print(f"  SE of difference : {se_diff:.4f}")
print(f"  t-statistic      : {t_diff:.4f}")
print(f"  p-value          : {p_diff:.4f}  {'*** significant' if p_diff < 0.05 else 'not significant'}")

# ─────────────────────────────────────────────
# 2. MANN-KENDALL EXACT TEST ON MEDIAN
# ─────────────────────────────────────────────

def mann_kendall_exact(x):
    def compute_s(seq):
        s = 0
        for i in range(len(seq) - 1):
            for j in range(i + 1, len(seq)):
                s += np.sign(seq[j] - seq[i])
        return s

    observed_s = compute_s(x)
    all_s      = [compute_s(perm) for perm in permutations(x)]
    p_exact    = sum(1 for s in all_s if abs(s) >= abs(observed_s)) / len(all_s)
    tau        = observed_s / (len(x) * (len(x) - 1) / 2)

    return observed_s, tau, p_exact


print("\n" + "=" * 55)
print("MANN-KENDALL EXACT TEST ON MEDIAN")
print("=" * 55)
for label, data in [("Native", native_median), ("Migrating", migrating_median)]:
    s, tau, p = mann_kendall_exact(data)
    direction = "downward" if tau < 0 else "upward"
    print(f"\n{label}")
    print(f"  S statistic : {s}")
    print(f"  Kendall tau : {tau:+.4f}  ({direction} trend)")
    print(f"  p-value     : {p:.4f}  {'*** significant' if p < 0.05 else 'not significant'}")

# ─────────────────────────────────────────────
# 3. COMPARISON SUMMARY: MEAN VS MEDIAN
# ─────────────────────────────────────────────

native_mean_vals   = np.array([41.865980, 45.332769, 44.491083, 41.682730, 40.224726, 40.813832])
migrating_mean_vals = np.array([90.610106, 89.488594, 81.937664, 77.234132, 79.641804, 70.922299])

n_slope_mean, _, _, _, n_p_mean, _ = weighted_regression(years, native_mean_vals, native_n)
m_slope_mean, _, _, _, m_p_mean, _ = weighted_regression(years, migrating_mean_vals, migrating_n)
_, _, n_mk_p_median = mann_kendall_exact(native_median)
_, _, m_mk_p_median = mann_kendall_exact(migrating_median)

# print("\n" + "=" * 55)
# print("COMPARISON SUMMARY: MEAN VS MEDIAN")
# print("=" * 55)
# print(f"\n{'':30s} {'Mean':>10s}  {'Median':>10s}")
# print(f"  {'Native — WLS slope (km/yr)':30s} {n_slope_mean:>+10.4f}  {native_slope:>+10.4f}")
# print(f"  {'Native — WLS p-value':30s} {n_p_mean:>10.4f}  {native_p:>10.4f}")
# print(f"  {'Native — MK p-value':30s} {n_mk_p_mean:>10.4f}  {n_mk_p_median:>10.4f}")
# print(f"  {'Migrating — WLS slope (km/yr)':30s} {m_slope_mean:>+10.4f}  {migr_slope:>+10.4f}")
# print(f"  {'Migrating — WLS p-value':30s} {m_p_mean:>10.4f}  {migr_p:>10.4f}")
# print(f"  {'Migrating — MK p-value':30s} {m_mk_p_mean:>10.4f}  {m_mk_p_median:>10.4f}")

WEIGHTED LINEAR REGRESSION ON MEDIAN

Native
  Slope       : -0.1881 km/year  (SE = 0.3083)
  Intercept   : 28.0480
  R²          : 0.0851
  t-statistic : -0.6101
  p-value     : 0.5748  not significant

Migrating
  Slope       : +0.0549 km/year  (SE = 0.0552)
  Intercept   : 35.7169
  R²          : 0.1977
  t-statistic : 0.9929
  p-value     : 0.3770  not significant

Slope difference (native − migrating): -0.2430 km/year
  SE of difference : 0.3132
  t-statistic      : -0.7756
  p-value          : 0.4603  not significant

MANN-KENDALL EXACT TEST ON MEDIAN

Native
  S statistic : -7.0
  Kendall tau : -0.4667  (downward trend)
  p-value     : 0.2722  not significant

Migrating
  S statistic : 7.0
  Kendall tau : +0.4667  (upward trend)
  p-value     : 0.2722  not significant


Here I have 2 robustness checks: the MK and using the median.


* Weighted OLS and MK:

They test fundamentally different things
The weighted regression asks: given I assume a linear trend, is the slope significantly different from zero? The result depends on the actual magnitudes of your means, the weights you chose, and the linearity assumption all being reasonable.
Mann-Kendall asks: does this sequence tend to go in one direction consistently? It doesn't care about linearity, doesn't care about the magnitudes, doesn't care about your sample-size weights at all. It just looks at the rank ordering of years.
So they have almost no shared assumptions, which is exactly what you want from a robustness check.

* median:

is less affected by outliers

#### interpretation: failed the robustness check

the median for migrants being flat while the mean declines sharply suggests there are high-distance  observations being pulled down over time while other migrating birds haven't changed behavior, but extreme observations have shifted. Any test on the mean will capture that extreme effect, not necessarily a population-wide shift. Using the median gives it a more robust picture.